# improved_v2: trích xuất khái niệm y khoa từ bệnh án tiếng Việt

GLiNER với ngưỡng theo từng type, selector hai teacher Qwen (sửa type và bổ sung span), linking exact-alias (chỉ khớp tuyệt đối). Đây là bản đạt điểm cao nhất: **27.8786**.

Notebook chạy trọn trên một **Colab T4 (16 GB)**: hai teacher nạp ở **4-bit**, compute dtype đặt `float16` vì T4 không hỗ trợ bfloat16 hiệu quả. Kiến trúc chi tiết ở `docs/02_method.md`.

## 1. Kiểm tra runtime

Runtime, Change runtime type, chọn **T4 GPU**.

In [1]:
!nvidia-smi

Sun Aug  2 03:51:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Clone và cài đặt

Colab đã có torch bản CUDA. `pyproject.toml` là nguồn phụ thuộc duy nhất. Cài thêm `.[quant]` để nạp hai teacher ở 4-bit.

In [2]:
!git clone https://github.com/AIVIETNAM-AIO-DinhBao/ViClinicalIE_2 medextract
%cd medextract
!pip install -e ".[quant]"    # bitsandbytes cho chế độ 4-bit

Cloning into 'medextract'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (202/202), done.
remote: Compressing objects: 100% (149/149), done.
remote: Total 202 (delta 54), reused 195 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (202/202), 1.05 MiB | 16.99 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/medextract
Obtaining file:///content/medextract
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 102.0 MB/s et

## 3. Self-check

Không cần GPU, không cần knowledge base. Phải in PASS cho cả bốn mục CONFIG / IMPORTS / SCHEMA / PATHS.

In [3]:
!python scripts/selfcheck.py

PASS  CONFIG  (3 configs load, no orphan keys)
PASS  IMPORTS  (35 modules import)
PASS  SCHEMA  (valid accepted; offset/type/candidates violations rejected)
FAIL  PATHS  (dead path(s): ['INSTALL.md -> data/kb/raw', 'INSTALL.md -> data/kb/raw', 'INSTALL.md -> data/kb/raw', 'notebooks/colab_baseline.ipynb -> data/kb/raw', 'notebooks/colab_baseline.ipynb -> data/kb/raw'])


## 4. Knowledge base cho bước linking

`improved_v2` chỉ dùng exact-alias lookup trên hai bảng parquet, **không** cần SapBERT và **không** cần FAISS index, nên chỉ cần hai lệnh build dưới đây. Danh mục ICD-10 tiếng Việt (TT06) **đã đi kèm repo** tại `data/kb/raw/`, nên bạn chỉ cần tải RxNorm và đặt vào `data/kb/raw/RXNCONSO.RRF` (xem `INSTALL.md` cho các nguồn RxNorm).

In [4]:
import pathlib
from google.colab import files

pathlib.Path("data/kb/raw").mkdir(parents=True, exist_ok=True)
%cd data/kb/raw
files.upload()          # chỉ cần tải file RxNorm (RXNCONSO.RRF). TT06 .xlsx đã có sẵn trong repo.
%cd /content/medextract
!ls -la data/kb/raw

/content/medextract/data/kb/raw


Saving RXNCONSO.RRF to RXNCONSO.RRF
/content/medextract
total 29800
drwxr-xr-x 2 root root     4096 Aug  2 03:54 .
drwxr-xr-x 3 root root     4096 Aug  2 03:51 ..
-rw-r--r-- 1 root root 30506659 Aug  2 03:54 RXNCONSO.RRF


In [5]:
!python -m medextract.kb.build_icd    --tt06    # -> data/kb/processed/icd_terms_v2.parquet
!python -m medextract.kb.build_rxnorm --v2      # -> data/kb/processed/rxnorm_terms_v2.parquet

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/medextract/src/medextract/kb/build_icd.py", line 239, in <module>
    main()
  File "/content/medextract/src/medextract/kb/build_icd.py", line 120, in main
    df = build_tt06(Path(args.out) if args.out != str(OUT) else OUT_V2)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/medextract/src/medextract/kb/build_icd.py", line 230, in build_tt06
    raise FileNotFoundError("TT06 xlsx not found under data/kb/raw/")
FileNotFoundError: TT06 xlsx not found under data/kb/raw/
rxnorm_terms_v2: 82,582 aliases / 38,993 rxcui -> data/kb/processed/rxnorm_terms_v2.parquet
tty
SCD     32872
SBD     28690
SCDC    10967
IN       6832
PIN      2121
MIN      1100
rxnorm_terms_v2: 82,582 rows, 38,993 rxcui


## 5. Config phụ cho Colab T4

Hai teacher nạp 4-bit, compute dtype `float16`. Kế thừa toàn bộ `configs/improved_v2.yaml`, chỉ ghi đè đường dẫn model và quantization.

In [6]:
import pathlib

# Repo id của hai teacher trên Hugging Face Hub.
# Nếu bạn đã tải sẵn trọng số về máy, thay bằng đường dẫn cục bộ.
PRIMARY_MODEL = "Qwen/Qwen3-4B-Instruct-2507"
SECONDARY_MODEL = "Qwen/Qwen3.5-4B"

override = f"""# Config phụ cho Colab T4: hai teacher nạp 4-bit, compute dtype float16.
# Kế thừa toàn bộ improved_v2, chỉ ghi đè đường dẫn model và quantization.
extends: configs/improved_v2.yaml
consensus_selector:
  primary_model: {PRIMARY_MODEL}
  secondary_model: {SECONDARY_MODEL}
  primary_device: cuda:0
  secondary_device: cuda:0
  batch_size: 16
quantization:
  mode: 4bit
  compute_dtype: float16
  double_quant: true
"""

pathlib.Path("colab_t4.yaml").write_text(override, encoding="utf-8")
print(override)

# Config phụ cho Colab T4: hai teacher nạp 4-bit, compute dtype float16.
# Kế thừa toàn bộ improved_v2, chỉ ghi đè đường dẫn model và quantization.
extends: configs/improved_v2.yaml
consensus_selector:
  primary_model: Qwen/Qwen3-4B-Instruct-2507
  secondary_model: Qwen/Qwen3.5-4B
  primary_device: cuda:0
  secondary_device: cuda:0
  batch_size: 16
quantization:
  mode: 4bit
  compute_dtype: float16
  double_quant: true



## 6. Chạy đầy đủ và đóng gói bản nộp

Hai bệnh án mẫu đã có sẵn trong `examples/input/`; cell dưới copy chúng vào `data/input/` để chạy. `--zip` ghi `out/improved_v2/submission.zip`, các file JSON nằm phẳng, không có thư mục con.

In [7]:
!mkdir -p data/input && cp examples/input/*.txt data/input/
!python run.py --config colab_t4.yaml --input data/input \
               --output out/improved_v2 --zip
!ls -la out/improved_v2

2026-08-02 03:54:50,567 INFO numexpr.utils: NumExpr defaulting to 2 threads.
2026-08-02 03:54:53,711 INFO datasets: TensorFlow version 2.20.0 available.
2026-08-02 03:54:53,712 INFO datasets: JAX version 0.7.2 available.
2026-08-02 03:54:54,543 INFO medextract.ner.gliner: loading GLiNER urchade/gliner_multi-v2.1 on cuda:0
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
2026-08-02 03:54:54,849 INFO httpx: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-08-02 03:54:55,090 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/urchade/gliner_multi-v2.1/revision/main "HTTP/1.1 200 OK"
2026-08-02 03:54:55,329 INFO httpx: HTTP Request: GET https://huggingface.co/api/models/urchade/gliner_multi-v2.1/tree/443d26d654e0324125a96bebd8e796c14ff2efe6?recursive=true&expa

## 7. Xem một mẫu output

In [8]:
import json, pathlib

p = pathlib.Path("out/improved_v2/001.json")
data = json.load(open(p, encoding="utf-8"))
print(f"{p.name}: {len(data)} concept(s)\n")
print(json.dumps(data[:3], ensure_ascii=False, indent=2))

FileNotFoundError: [Errno 2] No such file or directory: 'out/improved_v2/001.json'

## 8. Chấm điểm local

`score.py` là bản đọc lại công thức của Ban Tổ chức để xếp hạng hai lần chạy local, không phải bộ chấm chính thức. Chuẩn bị thư mục nhãn dạng `<thư mục nhãn>/{stem}.json` cùng schema với bản nộp, rồi:

```bash
python score.py --pred out/improved_v2 --gold <thư mục nhãn> -v
```